# Agent State Sandbox
Tests a custom `AgentState` with `tool_call_count` to limit tool calls gracefully.

In [ ]:
import sys
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')

## 1. Imports

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from deps import make_obj
from tool import document_search

## 2. Custom State

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    tool_call_count: int

print(AgentState.__annotations__)

## 3. Prompt + LLM

In [ ]:
AGENT_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are DiabeticAssist, a clinical reference chatbot specializing in diabetes care.
You answer questions using verified medical documents only.
Always cite the source document when referencing specific facts.
Never provide personal medical advice or recommend starting/stopping treatments.
Add a safety disclaimer when discussing dosages, insulin regimens, or drug interactions.
Focus on topics including: Type 1 and Type 2 diabetes, prediabetes, gestational diabetes,
blood glucose management, HbA1c targets, medications (metformin, insulin, GLP-1 agonists, SGLT2 inhibitors),
dietary guidance, and diabetes complications.
If a question is outside the scope of available documents, say so clearly.
Always call document_search at most once per question."""),
    MessagesPlaceholder("messages"),
])

tools = [document_search]
tool_node = ToolNode(tools)
llm = AGENT_PROMPT | make_obj().bind_tools(tools)
print("LLM ready")

## 4. Nodes
`call_model` increments the counter when the LLM wants to call a tool.  
`should_continue` routes to END once the counter hits 1 — no recursion errors.

In [ ]:
def call_model(state: AgentState):
    response = llm.invoke({"messages": state["messages"]})
    count = state.get("tool_call_count", 0)
    if response.tool_calls:
        count += 1
    print(f"[call_model] tool_call_count={count}, tool_call={bool(response.tool_calls)}")
    return {"messages": [response], "tool_call_count": count}

def should_continue(state: AgentState):
    last = state["messages"][-1]
    if last.tool_calls and state["tool_call_count"] < 1:
        print("[should_continue] -> tools")
        return "tools"
    print("[should_continue] -> END")
    return END

## 5. Build + Compile

In [ ]:
graph = StateGraph(AgentState)
graph.add_node("agent", call_model)
graph.add_node("tools", tool_node)
graph.set_entry_point("agent")
graph.add_conditional_edges("agent", should_continue)
graph.add_edge("tools", "agent")

agent = graph.compile()
print("Agent compiled")

## 6. Test — Tool Call Question

In [ ]:
response = await agent.ainvoke(
    {"messages": [HumanMessage(content="What are the HbA1c targets for type 2 diabetes?")],
     "tool_call_count": 0}
)

for msg in response["messages"]:
    print(type(msg).__name__, ":", msg.content[:300] if msg.content else msg.tool_calls)
    print()

print("tool_call_count:", response["tool_call_count"])

## 7. Test — No Tool Call Question

In [ ]:
response = await agent.ainvoke(
    {"messages": [HumanMessage(content="What is blue?")],
     "tool_call_count": 0}
)

for msg in response["messages"]:
    print(type(msg).__name__, ":", msg.content[:300] if msg.content else msg.tool_calls)
    print()

print("tool_call_count:", response["tool_call_count"])